# Generación del conjunto de datos experimental

In [2]:
# Importación de librerias
from pathlib import Path
import pandas as pd
import shutil
from tqdm import tqdm

## 1. Definición de rutas

Se definen las rutas correspondientes al dataset procesado, el manifiesto experimental y el directorio de salida del conjunto reorganizado.

In [3]:
PROJECT_ROOT = Path("../../")

DATASET_DIR = (
    PROJECT_ROOT 
    / "data" 
    / "processed" 
    / "MMOTU_OTU_2D"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "configs"
    / "splits"
    / "mmotu_experimental_split.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "MMOTU_OTU_2D_experimental"
)


DATASET_DIR, MANIFEST_PATH, OUTPUT_DIR

(WindowsPath('../../data/processed/MMOTU_OTU_2D'),
 WindowsPath('../../configs/splits/mmotu_experimental_split.csv'),
 WindowsPath('../../data/processed/MMOTU_OTU_2D_experimental'))

## 2. Carga del manifiesto experimental

El manifiesto contiene la información asociada a cada muestra, incluyendo:

- identificador de imagen,
- etiqueta binaria,
- clase binaria,
- partición experimental asignada.

In [4]:
df = pd.read_csv(MANIFEST_PATH)

df.head()

,image_id,image_name,source_split,original_class,binary_label,binary_class,experimental_split
0,789,789.JPG,train,5,0,benign,train_internal
1,312,312.JPG,train,4,0,benign,train_internal
2,1196,1196.JPG,train,6,1,malignant,train_internal
3,297,297.JPG,train,1,0,benign,train_internal
4,263,263.JPG,train,2,0,benign,train_internal


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   image_id            1469 non-null   int64
 1   image_name          1469 non-null   str  
 2   source_split        1469 non-null   str  
 3   original_class      1469 non-null   int64
 4   binary_label        1469 non-null   int64
 5   binary_class        1469 non-null   str  
 6   experimental_split  1469 non-null   str  
dtypes: int64(3), str(4)
memory usage: 80.5 KB


## 3. Distribución de muestras por partición experimental

Se verifica la cantidad de muestras asignadas a cada subconjunto antes de generar la estructura física del dataset.

In [6]:
df["experimental_split"].value_counts()

experimental_split
train_internal      800
final_validation    469
val_internal        200
Name: count, dtype: int64

In [7]:
pd.crosstab(df["experimental_split"], df["binary_class"])

binary_class,benign,malignant
experimental_split,,
final_validation,421,48
train_internal,713,87
val_internal,178,22


## 4. Creación de estructura de salida

Se crea la estructura de directorios correspondiente al conjunto experimental:

In [9]:
splits = {
    "train_internal": "train",
    "val_internal": "validation",
    "final_validation": "test"
}


for split_folder in splits.values():
    (OUTPUT_DIR / split_folder / "images").mkdir(
        parents=True,
        exist_ok=True
    )
    
    (OUTPUT_DIR / split_folder / "masks").mkdir(
        parents=True,
        exist_ok=True
    )

print("Estructura creada correctamente")

Estructura creada correctamente


## 5. Copia de imágenes y máscaras

Cada muestra es ubicada en el subconjunto correspondiente según la partición definida en el manifiesto experimental.

Se conserva la relación imagen-máscara requerida para la etapa de segmentación y se mantiene la información necesaria para la clasificación binaria.

In [10]:
SOURCE_IMAGES = {
    "train": DATASET_DIR / "train" / "images",
    "validation": DATASET_DIR / "train" / "images",
    "test": DATASET_DIR / "validation" / "images"
}

SOURCE_MASKS = {
    "train": DATASET_DIR / "train" / "masks",
    "validation": DATASET_DIR / "train" / "masks",
    "test": DATASET_DIR / "validation" / "masks"
}

In [17]:
for _, row in tqdm(df.iterrows(), total=len(df)):

    exp_split = row["experimental_split"]
    target_split = splits[exp_split]
    
    image_name = row["image_name"]
    mask_name = Path(image_name).stem + "_binary.PNG"
    
    image_source = (SOURCE_IMAGES[target_split] / image_name)
    mask_source = (SOURCE_MASKS[target_split] / mask_name)
    
    image_target = (OUTPUT_DIR / target_split / "images" / image_name)
    mask_target = (OUTPUT_DIR / target_split / "masks" / mask_source.name)
    
    shutil.copy2(image_source, image_target)    
    shutil.copy2(mask_source, mask_target)

100%|██████████| 1469/1469 [00:28<00:00, 52.01it/s] 


## 6. Verificación del conjunto generado

Se verifica que cada subconjunto contenga la cantidad esperada de imágenes y máscaras, asegurando la correspondencia entre ambos archivos.

In [18]:
for split in ["train", "validation", "test"]:
    
    images = list((OUTPUT_DIR / split / "images").glob("*"))
    masks = list((OUTPUT_DIR / split / "masks").glob("*"))
    
    print(split)
    print("Images:", len(images))
    print("Masks :", len(masks))
    print("-"*30)

train
Images: 800
Masks : 800
------------------------------
validation
Images: 200
Masks : 200
------------------------------
test
Images: 469
Masks : 469
------------------------------
